# 03 · Unsupervised Learning and AutoML

Compare clustering assumptions, test stability, inspect PCA/t-SNE/UMAP, and preserve complete Grid/Random/Optuna search histories without preprocessing leakage.

This is a learning and interview notebook, not a thin launcher. It combines mechanism,
executable evidence, diagnostics, limitations, and experiments. Run from top to bottom;
keep `RUN_TRAINING=False` to inspect committed/CI results, or turn it on to reproduce them.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Learning objectives

- Explain the geometry assumed by K-Means, DBSCAN, and agglomerative clustering.
- Treat silhouette as one diagnostic rather than ground truth.
- Distinguish PCA's linear variance objective from t-SNE/UMAP neighborhood visualizations.
- Measure cluster stability across restarts.
- Compare exhaustive, random, and Bayesian/TPE search with a finite budget.

In [ ]:
if not (DATA_DIR / "manifest.json").exists():
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--profile", "ci", "--output", str(DATA_DIR)],
        check=True,
    )
subprocess.run([sys.executable, "scripts/validate_data.py", "--data-dir", str(DATA_DIR)], check=True)
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(manifest["dataset_version"], manifest["rows"])

In [ ]:
metric_path = METRICS_DIR / "unsupervised.json"
if RUN_TRAINING or not metric_path.exists():
    metric_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "scripts/run_unsupervised.py", "--data-dir", str(DATA_DIR), "--profile", "ci", "--output", str(METRICS_DIR / "unsupervised.json")], check=True)
result = json.loads(metric_path.read_text())

print("loaded", metric_path, "dataset", result.get("_run", {}).get("dataset_version"))

In [ ]:
import pandas as pd
clustering = pd.DataFrame(result["unsupervised"]["clustering"]).T
display(clustering)
display(pd.DataFrame(result["automl"]["leaderboard"]))

## Why scaling comes first

Distance-based methods are dominated by large-scale columns. Median imputation and StandardScaler
are fitted before clustering. In supervised CV they must be inside the Pipeline; unsupervised
exploration still needs a clearly documented fit population.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

customers = pd.read_csv(DATA_DIR / "customers.csv.gz")
columns = ["age", "tenure_days", "income", "visits_30d", "support_tickets", "loyalty_score"]
raw = SimpleImputer(strategy="median").fit_transform(customers[columns])
scaled = StandardScaler().fit_transform(raw)
display(pd.DataFrame({"raw_std": raw.std(0), "scaled_std": scaled.std(0)}, index=columns))

## K-Means objective

K-Means alternates assignment and centroid updates to minimize within-cluster squared distance.
It favors convex, similarly scaled clusters and requires $k$ in advance.

In [ ]:
import numpy as np

def one_kmeans_step(values, centers):
    distances = ((values[:, None, :] - centers[None, :, :]) ** 2).sum(2)
    labels = distances.argmin(1)
    new_centers = np.vstack([values[labels == index].mean(0) for index in range(len(centers))])
    return labels, new_centers

demo = scaled[:100, :2]
labels, centers = one_kmeans_step(demo, demo[[0, 25, 50, 75]])
print("one assignment/update step", np.bincount(labels), centers.shape)

## DBSCAN and hierarchy

DBSCAN can mark sparse points as noise and discover non-convex shapes, but `eps` is scale-sensitive.
Agglomerative clustering builds a merge hierarchy; this experiment cuts it at four clusters.

In [ ]:
for name, row in result["unsupervised"]["clustering"].items():
    print(name, "silhouette", row["silhouette"], "noise", row["noise_rate"], "sizes", row["cluster_sizes"])

## Stability

A visually pleasing clustering can be unstable. Five K-Means restarts are compared with pairwise
Adjusted Rand Index (ARI), which is permutation-invariant and chance-adjusted.

In [ ]:
stability = result["unsupervised"]["stability"]
display(pd.Series(stability["pairwise_ari"]).describe())
pd.Series(stability["pairwise_ari"]).plot.hist(bins=8, title="Pairwise ARI across restarts")
plt.show()

## PCA from linear algebra

After centering, PCA directions are right singular vectors of the data matrix. The first components
maximize explained variance but need not preserve nonlinear neighborhoods.

In [ ]:
centered = scaled - scaled.mean(0)
_, singular_values, vectors = np.linalg.svd(centered, full_matrices=False)
explained = singular_values**2 / (len(centered) - 1)
ratio = explained / explained.sum()
print("manual first-two explained variance", ratio[:2].sum())
print("recorded PCA result", result["unsupervised"]["dimension_reduction"]["pca_explained_variance"])

In [ ]:
from IPython.display import Image, display
display(Image(filename="artifacts/figures/dimension-reduction-ci.png"))

## Do not over-interpret t-SNE/UMAP

Axes have no direct business meaning, global distances can be distorted, and different seeds or
hyperparameters may change the picture. Use embeddings to generate hypotheses, not labels.

## AutoML search strategies

Grid is exhaustive over a small declared set. Random search spends budget on sampled combinations.
Optuna TPE uses previous trials to bias future proposals. All three still require a valid CV design.

In [ ]:
histories = []
for search in ("grid_search", "random_search", "optuna_tpe"):
    for row in result["automl"][search]["history"]:
        histories.append({"search": search, "score": row.get("mean_cv_roc_auc", row.get("value")), "params": str(row["params"])})
history = pd.DataFrame(histories)
display(history)
history.assign(trial=history.groupby("search").cumcount()).pivot(index="trial", columns="search", values="score").plot(marker="o")
plt.ylabel("CV ROC-AUC")
plt.show()

In [ ]:
print("Leakage control:", result["automl"]["leakage_control"])
print("Winner:", result["automl"]["leaderboard"][0])

## Exercises and interview checkpoint

1. Sweep DBSCAN `eps` and plot clusters/noise/silhouette together.
2. Repeat stability with bootstrap samples rather than seeds only.
3. Compare PCA before versus after scaling.
4. Explain why choosing the best CV score still has selection bias.
5. Add a nested-CV outer loop and report the difference.